# YOLO11 균열 세그멘테이션 모델 열기 및 테스트

다른 컴퓨터와 새로운 Google Drive 계정에서 실행하는 Colab 노트북입니다. **위에서부터 순서대로 실행**하세요.

준비물: 기존 컴퓨터에서 다운로드한 `best.pt` 파일

## 1. 필수 패키지 설치

In [ ]:
!pip install -U ultralytics -q

## 2. 새로운 Google Drive 계정 연결
계정 선택 화면에서 앞으로 사용할 새 Google 계정을 선택하세요.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive')
print('Google Drive 연결:', DRIVE_ROOT.exists())
print('내 드라이브:', DRIVE_ROOT)

## 3. `best.pt`를 새 Google Drive에 등록

새 Drive의 `YOLO_Models/YOLO11s_Crack_100epochs/best.pt`를 먼저 확인합니다. 파일이 없으면 업로드 창이 열립니다. 기존 컴퓨터에서 가져온 `best.pt`를 선택하세요.

In [ ]:
from google.colab import files
from pathlib import Path
import shutil

MODEL_DIR = DRIVE_ROOT / 'YOLO_Models' / 'YOLO11s_Crack_100epochs'
MODEL_PATH = MODEL_DIR / 'best.pt'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

if MODEL_PATH.exists():
    print('기존 모델 발견:', MODEL_PATH)
else:
    print('새 Drive에 모델이 없습니다. best.pt 파일을 선택하세요.')
    uploaded = files.upload()
    pt_files = [Path('/content') / name for name in uploaded if name.lower().endswith('.pt')]
    if not pt_files:
        raise ValueError('.pt 모델 파일이 업로드되지 않았습니다.')
    shutil.copy2(pt_files[0], MODEL_PATH)
    print('새 Google Drive에 모델 저장 완료:', MODEL_PATH)

print('모델 존재:', MODEL_PATH.exists())
print('모델 용량:', round(MODEL_PATH.stat().st_size / 1024**2, 1), 'MB')

## 4. 모델 불러오기 및 확인

In [ ]:
from ultralytics import YOLO

model = YOLO(str(MODEL_PATH))
print('모델 로드 완료')
print('작업 유형:', model.task)
print('클래스:', model.names)

## 5. GPU 확인

In [ ]:
import torch

DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print('사용 장치:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('DEVICE:', DEVICE)

## 6. 테스트할 균열 사진 한 장 업로드

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
image_names = [name for name in uploaded if Path(name).suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}]
if not image_names:
    raise ValueError('이미지 파일이 업로드되지 않았습니다.')
IMAGE_PATH = Path('/content') / image_names[0]
print('테스트 이미지:', IMAGE_PATH)

## 7. 균열 마스크 예측

이 모델은 640 해상도로 학습했으므로 먼저 `imgsz=640`으로 테스트합니다. 균열을 놓치면 `conf=0.15` 정도로 낮춰 비교할 수 있습니다.

In [ ]:
results = model.predict(
    source=str(IMAGE_PATH),
    imgsz=640,
    conf=0.20,
    iou=0.70,
    device=DEVICE,
    retina_masks=True,
    verbose=False
)

result = results[0]
box_count = 0 if result.boxes is None else len(result.boxes)
mask_count = 0 if result.masks is None else len(result.masks.data)
print('탐지된 균열 수:', box_count)
print('생성된 마스크 수:', mask_count)
if result.boxes is not None and len(result.boxes):
    print('신뢰도:', [round(float(x), 3) for x in result.boxes.conf.cpu()])

## 8. 예측 결과 확인

In [ ]:
import cv2
import matplotlib.pyplot as plt

prediction_bgr = result.plot()
prediction_rgb = cv2.cvtColor(prediction_bgr, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(14, 10))
plt.imshow(prediction_rgb)
plt.axis('off')
plt.title('YOLO11s Crack Segmentation Result')
plt.show()

## 9. 결과를 새로운 Google Drive에 저장

In [ ]:
from datetime import datetime

SAVE_DIR = DRIVE_ROOT / 'YOLO11_Crack_Test_Results'
SAVE_DIR.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
SAVE_PATH = SAVE_DIR / f'{IMAGE_PATH.stem}_prediction_{timestamp}.jpg'
success = cv2.imwrite(str(SAVE_PATH), prediction_bgr)
if not success:
    raise IOError(f'결과 이미지 저장에 실패했습니다: {SAVE_PATH}')
print('Google Drive 저장 완료:', SAVE_PATH)

## 참고

- 이 노트북은 사진 한 장의 육안 검사용입니다. 정량적인 IoU·mAP 검증에는 정답 마스크가 포함된 별도의 test 데이터셋이 필요합니다.
- 이후 다시 실행할 때는 새 Drive에 저장된 `best.pt`를 자동으로 불러오므로 모델을 다시 업로드하지 않아도 됩니다.